In [51]:
import pandas as pd
import numpy as np
import plotly.express as px
import folium
from folium.plugins import HeatMap


In [52]:
district_df = pd.read_csv(
    "../data/cleaned_merged_output_final.csv",
    sep=";",
    encoding="utf-8-sig"
)

origin_df = pd.read_csv(
    "../data/country-by-origin-cleaned.csv",
    encoding="utf-8-sig"
)

In [53]:
# District population data

year_cols = [str(y) for y in range(1995, 2026)]

population_long = district_df.melt(
    id_vars=["age_group", "sex", "ancestry", "district"],
    value_vars=year_cols,
    var_name="year",
    value_name="population"
)

population_long["year"] = population_long["year"].astype(int)
population_long["population"] = pd.to_numeric(population_long["population"], errors="coerce")

population_long.head()

,age_group,sex,ancestry,district,year,population
0,Years in total,Sex total,Ancestry total,Copenhagen total,1995,469863.0
1,Years in total,Sex total,Ancestry total,Indre By,1995,41340.0
2,Years in total,Sex total,Ancestry total,Østerbro,1995,63207.0
3,Years in total,Sex total,Ancestry total,Nørrebro,1995,68508.0
4,Years in total,Sex total,Ancestry total,Vesterbro/Kongens Enghave,1995,47336.0


In [54]:
# Visualization 1 — Copenhagen population growth

total_pop = population_long[
    (population_long["district"] == "Copenhagen total") &
    (population_long["ancestry"] == "Ancestry total")
]

fig = px.line(
    total_pop,
    x="year",
    y="population",
    markers=True,
    title="Copenhagen's population has grown steadily since 1995",
    labels={
        "year": "Year",
        "population": "Population"
    }
)

fig.show()

This visualization introduces the overall story by showing how Copenhagen’s population has changed from 1995 to 2025. A line chart is appropriate because the main focus is development over time.

In [55]:
# Visualization 2 — Population by district

main_districts = [
    "Indre By",
    "Østerbro",
    "Nørrebro",
    "Vesterbro/Kongens Enghave",
    "Valby",
    "Vanløse",
    "Brønshøj-Husum",
    "Bispebjerg",
    "Amager Øst",
    "Amager Vest"
]

district_pop = population_long[
    (population_long["district"].isin(main_districts)) &
    (population_long["ancestry"] == "Ancestry total")
]

fig = px.line(
    district_pop,
    x="year",
    y="population",
    color="district",
    title="Population development across Copenhagen districts",
    labels={
        "year": "Year",
        "population": "Population",
        "district": "District"
    }
)

fig.show()

This interactive line chart lets the reader compare how different districts developed over time. This is important because the project focuses on whether Copenhagen’s demographic change happened evenly across the city or was concentrated in certain areas.

In [56]:
# Visualization 3 — International population share by district

international_groups = [
    "Immigrants of western origin",
    "Descendats of western origin",
    "Immigrants of non-western origin",
    "Descendants of non-western origin"
]

district_ancestry = population_long[
    population_long["district"].isin(main_districts)
]

international = (
    district_ancestry[district_ancestry["ancestry"].isin(international_groups)]
    .groupby(["district", "year"], as_index=False)["population"]
    .sum()
    .rename(columns={"population": "international_population"})
)

total = (
    district_ancestry[district_ancestry["ancestry"] == "Ancestry total"]
    [["district", "year", "population"]]
    .rename(columns={"population": "total_population"})
)

international_share = international.merge(total, on=["district", "year"])
international_share["international_share"] = (
    international_share["international_population"] /
    international_share["total_population"] * 100
)

fig = px.line(
    international_share,
    x="year",
    y="international_share",
    color="district",
    title="Share of residents with immigrant or descendant background by district",
    labels={
        "year": "Year",
        "international_share": "Share of population (%)",
        "district": "District"
    }
)

fig.show()

This visualization is central to the story because it shows how “international” different Copenhagen districts became over time. Instead of directly using political labels such as “ghetto”, the chart shows measurable demographic differences between districts.

In [57]:
# Visualization 4 — Which origin countries grew the most?

origin_total = (
    origin_df
    .groupby(["country", "year"], as_index=False)["population"]
    .sum()
)

origin_1995 = origin_total[origin_total["year"] == 1995][["country", "population"]]
origin_2025 = origin_total[origin_total["year"] == 2025][["country", "population"]]

growth = origin_2025.merge(
    origin_1995,
    on="country",
    suffixes=("_2025", "_1995")
)

growth["absolute_growth"] = growth["population_2025"] - growth["population_1995"]

top_growth = growth.sort_values("absolute_growth", ascending=False).head(15)

fig = px.bar(
    top_growth,
    x="absolute_growth",
    y="country",
    orientation="h",
    title="Countries of origin with the largest population growth in Copenhagen",
    labels={
        "absolute_growth": "Population growth, 1995–2025",
        "country": "Country of origin"
    }
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

This bar chart shows which country-of-origin groups grew the most in Copenhagen. It supports the broader story that Copenhagen has become more international, while keeping this city-level dataset separate from the district-level analysis.

In [58]:
# Heatmap of District vs Year. Share of international population

international_groups = [
    "Immigrants of western origin",
    "Descendants of western origin",
    "Immigrants of non-western origin",
    "Descendants of non-western origin"
]

district_data = population_long[
    population_long["district"] != "Copenhagen total"
]

# International population
international = (
    district_data[
        district_data["ancestry"].isin(international_groups)
    ]
    .groupby(["district", "year"], as_index=False)["population"]
    .sum()
)

# Total population
total = (
    district_data[
        district_data["ancestry"] == "Ancestry total"
    ][["district", "year", "population"]]
    .rename(columns={"population": "total_population"})
)

# Merge
heatmap_df = international.merge(
    total,
    on=["district", "year"]
)

# Calculate share
heatmap_df["share"] = (
    heatmap_df["population"] /
    heatmap_df["total_population"]
) * 100

In [59]:
pivot_df = heatmap_df.pivot(
    index="district",
    columns="year",
    values="share"
)

fig = px.imshow(
    pivot_df,
    aspect="auto",
    color_continuous_scale="Reds",
    title="Share of residents with immigrant or descendant background by district"
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="District"
)

fig.show()

The heatmap was chosen because it allows the reader to identify spatial and temporal patterns simultaneously. Instead of comparing many separate line charts, the heatmap highlights which districts experienced the strongest demographic changes and when these changes accelerated.

This visualization supports the project’s broader discussion about uneven demographic development across Copenhagen districts.

In [60]:
# District coordinates

district_coords = pd.DataFrame({
    "district": [
        "Indre By", "Østerbro", "Nørrebro", "Vesterbro/Kongens Enghave",
        "Valby", "Vanløse", "Brønshøj-Husum", "Bispebjerg",
        "Amager Øst", "Amager Vest"
    ],
    "lat": [
        55.6840, 55.7047, 55.6996, 55.6680,
        55.6624, 55.6872, 55.7078, 55.7182,
        55.6627, 55.6456
    ],
    "lon": [
        12.5900, 12.5770, 12.5481, 12.5450,
        12.5167, 12.4915, 12.4982, 12.5337,
        12.6208, 12.5836
    ]
})

In [61]:
# International share hotspot map

international_groups = [
    "Immigrants of western origin",
    "Descendats of western origin",
    "Immigrants of non-western origin",
    "Descendants of non-western origin"
]

international_2025 = (
    population_long[
        (population_long["year"] == 2025) &
        (population_long["ancestry"].isin(international_groups))
    ]
    .groupby("district", as_index=False)["population"]
    .sum()
    .rename(columns={"population": "international_population"})
)

total_2025 = (
    population_long[
        (population_long["year"] == 2025) &
        (population_long["ancestry"] == "Ancestry total")
    ][["district", "population"]]
    .rename(columns={"population": "total_population"})
)

international_map_df = international_2025.merge(total_2025, on="district")
international_map_df["international_share"] = (
    international_map_df["international_population"] /
    international_map_df["total_population"] * 100
)

international_map_df = international_map_df.merge(district_coords, on="district")

m1 = folium.Map(
    location=[55.6761, 12.5683],
    zoom_start=11,
    tiles="OpenStreetMap"
)

heat_data_1 = international_map_df[
    ["lat", "lon", "international_share"]
].values.tolist()

HeatMap(
    heat_data_1,
    radius=45,
    blur=35,
    min_opacity=0.4,
    max_zoom=13
).add_to(m1)

for _, row in international_map_df.iterrows():
    folium.Marker(
        location=[row["lat"], row["lon"]],
        tooltip=row["district"],
        popup=f"""
        <b>{row['district']}</b><br>
        International share: {row['international_share']:.1f}%<br>
        International population: {row['international_population']:,.0f}<br>
        Total population: {row['total_population']:,.0f}
        """
    ).add_to(m1)

m1

In [63]:
# International hotspot map — heatmap only

m1 = folium.Map(
    location=[55.6761, 12.5683],
    zoom_start=11,
    tiles="OpenStreetMap"
)

heat_data_1 = international_map_df[
    ["lat", "lon", "international_share"]
].values.tolist()

HeatMap(
    heat_data_1,
    radius=45,
    blur=35,
    min_opacity=0.45,
    max_zoom=13
).add_to(m1)

m1

In [64]:
# Income map

income_cols = [c for c in district_df.columns if c.startswith("avg_income_dkk_")]

income_long = district_df.melt(
    id_vars=["age_group", "sex", "ancestry", "district"],
    value_vars=income_cols,
    var_name="year",
    value_name="avg_income_dkk"
)

income_long["year"] = (
    income_long["year"]
    .str.replace("avg_income_dkk_", "", regex=False)
    .astype(int)
)

income_2024 = income_long[
    (income_long["year"] == 2024) &
    (income_long["district"].isin(district_coords["district"])) &
    (income_long["age_group"] == "Years in total") &
    (income_long["sex"] == "Sex total") &
    (income_long["ancestry"] == "Ancestry total")
].copy()

income_2024 = income_2024.merge(district_coords, on="district")

m2 = folium.Map(
    location=[55.6761, 12.5683],
    zoom_start=11,
    tiles="OpenStreetMap"
)

for _, row in income_2024.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=row["avg_income_dkk"] / 30000,
        popup=f"""
        <b>{row['district']}</b><br>
        Average income 2024: {row['avg_income_dkk']:,.0f} DKK
        """,
        tooltip=row["district"],
        fill=True,
        fill_opacity=0.7
    ).add_to(m2)

m2

In [65]:
# Income heatmap

m2 = folium.Map(
    location=[55.6761, 12.5683],
    zoom_start=11,
    tiles="OpenStreetMap"
)

# Heatmap data
heat_data_income = income_2024[
    ["lat", "lon", "avg_income_dkk"]
].values.tolist()

# Add heat layer
HeatMap(
    heat_data_income,
    radius=40,
    blur=30,
    min_opacity=0.45,
    max_zoom=13
).add_to(m2)

# Optional subtle district points
for _, row in income_2024.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=4,
        color="white",
        weight=1,
        fill=True,
        fill_color="white",
        fill_opacity=0.7,
        tooltip=f"""
        {row['district']}<br>
        Income: {row['avg_income_dkk']:,.0f} DKK
        """
    ).add_to(m2)

m2

In [66]:
# Population change map, 1995 → 2025

pop_1995 = population_long[
    (population_long["year"] == 1995) &
    (population_long["ancestry"] == "Ancestry total") &
    (population_long["district"].isin(district_coords["district"]))
][["district", "population"]].rename(columns={"population": "population_1995"})

pop_2025 = population_long[
    (population_long["year"] == 2025) &
    (population_long["ancestry"] == "Ancestry total") &
    (population_long["district"].isin(district_coords["district"]))
][["district", "population"]].rename(columns={"population": "population_2025"})

change_map_df = pop_1995.merge(pop_2025, on="district")
change_map_df["absolute_change"] = (
    change_map_df["population_2025"] - change_map_df["population_1995"]
)
change_map_df["percent_change"] = (
    change_map_df["absolute_change"] / change_map_df["population_1995"] * 100
)

change_map_df = change_map_df.merge(district_coords, on="district")

m3 = folium.Map(
    location=[55.6761, 12.5683],
    zoom_start=11,
    tiles="OpenStreetMap"
)

for _, row in change_map_df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=max(row["absolute_change"] / 2500, 5),
        popup=f"""
        <b>{row['district']}</b><br>
        Population 1995: {row['population_1995']:,.0f}<br>
        Population 2025: {row['population_2025']:,.0f}<br>
        Change: {row['absolute_change']:,.0f}<br>
        Percent change: {row['percent_change']:.1f}%
        """,
        tooltip=row["district"],
        fill=True,
        fill_opacity=0.7
    ).add_to(m3)

m3

In [67]:
# Population change heatmap

m3 = folium.Map(
    location=[55.6761, 12.5683],
    zoom_start=11,
    tiles="OpenStreetMap"
)

# Heatmap data
heat_data_change = change_map_df[
    ["lat", "lon", "absolute_change"]
].values.tolist()

# Add heat layer
HeatMap(
    heat_data_change,
    radius=45,
    blur=35,
    min_opacity=0.45,
    max_zoom=13
).add_to(m3)

# Optional subtle district markers
for _, row in change_map_df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=4,
        color="white",
        weight=1,
        fill=True,
        fill_color="white",
        fill_opacity=0.7,
        tooltip=f"""
        {row['district']}<br>
        Change: {row['absolute_change']:,.0f}<br>
        Percent change: {row['percent_change']:.1f}%
        """
    ).add_to(m3)

m3